# Submission Ensemble Visualization

대표 submission/dev 결과를 정리하고, 실제로 확인된 ensemble 개선을 시각화합니다.

메모:
- `MVCAF backbone test v1.1/v1.2`, `baseline v2.2.1`은 직접 확인한 dev loss를 사용했습니다.
- `dann v1.0.1`, `teacher regularization v1.2.0`은 checkpoint에 저장된 dev metric은 확인했지만, 당시 quick ensemble 탐색에는 구조 차이 때문에 바로 포함하지 못했습니다.
- 그래서 이 notebook에서는 `ensemble search에 실제 반영된 값`과 `참고용 checkpoint metric`을 구분해서 보여줍니다.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()


In [ ]:
evaluated_single_df = pd.DataFrame([
    {
        'model': 'MVCAF v1.1\nSwin Tiny',
        'group': 'evaluated_single',
        'dev_logloss': 0.08818083111605057,
        'source': 'direct dev evaluation',
    },
    {
        'model': 'MVCAF v1.2\nConvNeXt Small',
        'group': 'evaluated_single',
        'dev_logloss': 0.0976244062374926,
        'source': 'direct dev evaluation',
    },
    {
        'model': 'baseline v2.2.1',
        'group': 'evaluated_single',
        'dev_logloss': 0.1959295062036901,
        'source': 'direct dev evaluation',
    },
]).sort_values('dev_logloss').reset_index(drop=True)

reference_df = pd.DataFrame([
    {
        'model': 'teacher regularization v1.2.0',
        'group': 'reference_checkpoint',
        'dev_logloss': 0.05755749548375395,
        'source': 'checkpoint saved metric',
    },
    {
        'model': 'dann v1.0.1',
        'group': 'reference_checkpoint',
        'dev_logloss': 0.1217593184384174,
        'source': 'checkpoint saved metric',
    },
]).sort_values('dev_logloss').reset_index(drop=True)

pair_df = pd.DataFrame([
    {
        'ensemble': 'Swin v1.1 + ConvNeXt Small v1.2',
        'dev_logloss': 0.078902,
        'best_single_logloss': 0.08818083111605057,
    },
    {
        'ensemble': 'baseline v2.2.1 + ConvNeXt Small v1.2',
        'dev_logloss': 0.095960,
        'best_single_logloss': 0.0976244062374926,
    },
])
pair_df['improvement'] = pair_df['best_single_logloss'] - pair_df['dev_logloss']
pair_df = pair_df.sort_values('dev_logloss').reset_index(drop=True)

triple_df = pd.DataFrame([
    {
        'ensemble': 'baseline v2.2.1 + Swin v1.1 + ConvNeXt Small v1.2',
        'dev_logloss': 0.078926,
        'best_single_logloss': 0.08818083111605057,
    }
])
triple_df['improvement'] = triple_df['best_single_logloss'] - triple_df['dev_logloss']

display(evaluated_single_df)
display(reference_df)
display(pair_df)
display(triple_df)


## Notes

- `teacher regularization v1.2.0`과 `dann v1.0.1`은 빠진 게 아니라, 당시 ensemble search용 quick loader가 `baseline/MVCAF` 구조만 바로 처리해서 탐색 대상에서 임시 제외됐었습니다.
- 하지만 checkpoint 내부 metric은 살아 있어서 참고값으로는 넣었습니다.
- 즉, 아래 pair/triple 개선은 `실제로 직접 조합 계산한 결과`이고, `teacher/dann`은 아직 그 조합 계산에 포함된 값이 아닙니다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=evaluated_single_df,
    x='dev_logloss',
    y='model',
    hue='model',
    dodge=False,
    palette='Blues_r',
    legend=False,
    ax=ax,
)
ax.set_title('Single Model Dev Logloss (Directly Evaluated)')
ax.set_xlabel('dev logloss (lower is better)')
ax.set_ylabel('')
for i, v in enumerate(evaluated_single_df['dev_logloss']):
    ax.text(v + 0.002, i, f'{v:.6f}', va='center')
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(
    data=reference_df,
    x='dev_logloss',
    y='model',
    hue='model',
    dodge=False,
    palette='Oranges_r',
    legend=False,
    ax=ax,
)
ax.set_title('Reference Checkpoint Metrics (Not Yet Included in Ensemble Search)')
ax.set_xlabel('dev logloss (lower is better)')
ax.set_ylabel('')
for i, v in enumerate(reference_df['dev_logloss']):
    ax.text(v + 0.002, i, f'{v:.6f}', va='center')
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=pair_df,
    x='dev_logloss',
    y='ensemble',
    hue='ensemble',
    dodge=False,
    palette='Greens_r',
    legend=False,
    ax=ax,
)
ax.set_title('Pair Ensemble Dev Logloss')
ax.set_xlabel('dev logloss (lower is better)')
ax.set_ylabel('')
for i, row in pair_df.iterrows():
    ax.text(
        row['dev_logloss'] + 0.0015,
        i,
        f"{row['dev_logloss']:.6f}  (improve {row['improvement']:.6f})",
        va='center',
    )
plt.tight_layout()


In [ ]:
summary_df = pd.DataFrame([
    {'type': 'best single', 'name': 'MVCAF v1.1 Swin Tiny', 'dev_logloss': 0.08818083111605057},
    {'type': 'best pair', 'name': 'Swin v1.1 + ConvNeXt Small v1.2', 'dev_logloss': 0.078902},
    {'type': 'best triple', 'name': 'baseline + Swin + ConvNeXt Small', 'dev_logloss': 0.078926},
])

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=summary_df,
    x='dev_logloss',
    y='type',
    hue='type',
    dodge=False,
    palette=['#4C78A8', '#54A24B', '#F58518'],
    legend=False,
    ax=ax,
)
ax.set_title('Best Single vs Ensemble')
ax.set_xlabel('dev logloss (lower is better)')
ax.set_ylabel('')
for i, row in summary_df.iterrows():
    ax.text(row['dev_logloss'] + 0.0015, i, f"{row['dev_logloss']:.6f}\n{row['name']}", va='center')
plt.tight_layout()


## Summary

- 직접 확인된 single 모델 중 가장 좋은 것은 `MVCAF backbone test v1.1 / Swin Tiny`
- 직접 확인된 pair ensemble 중 가장 좋은 것은 `v1.1 Swin Tiny + v1.2 ConvNeXt Small`
- triple ensemble도 개선되지만, 사실상 핵심은 위 pair 조합
- 참고용 checkpoint metric만 보면 `teacher regularization v1.2.0`은 매우 강한 후보라, 이후 ensemble 탐색에 꼭 다시 포함할 가치가 있음
